<a href="https://colab.research.google.com/github/junseo25/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026_09_23_Jun_Seo_Lee_%E2%80%94_Cleaning_Clinic_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

   order_id           item   category  qty   price                   ts
0         1   Cheeseburger       Food  2.0   $7.50  2026-09-05T12:03:00
1         1   Cheeseburger       Food  2.0   $7.50  2026-09-05T12:03:00
2         2  cheese burger       food  1.0     7.5     09/05/2026 12:40
3         3    Foam Finger      Merch  NaN      12  2026-09-05 13:00:00
4         4   UVA T-Shirt     Apparel  2.0  $24.00     2026-09-05 13:05
5         5    Rain Poncho   RainGear -3.0       6  2026-09-05T13:20:00
6         6    rain poncho  rain-gear  4.0   $6.00                  NaN
7         7            NaN      Merch  1.0      12  2026-09-05T14:00:00

### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [3]:
# Inspect shape, types, missing values, and duplicates before making decisions.
print('shape:', df.shape)
print('\ntypes:')
print(df.dtypes)
print('\nnull values:')
print(df.isnull().sum())
print('\nduplicated rows:', df.duplicated().sum())

shape: (8, 6)

types:
order_id      int64
item            str
category        str
qty         float64
price           str
ts              str
dtype: object

null values:
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

duplicated rows: 1


**What is wrong with this data?** List at least five specific problems:

1. Missing item names
2. Inconsistent item names
3. Inconsistent category names
4. Price type as object instead of numerical data
5. Timestamp type as object instead of datetime
6. Invalid/missing data entries
7. Duplicate row(s)

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [4]:
removed = int(df.duplicated().sum())
clean = df.drop_duplicates().copy()
print('rows:', len(df), '->', len(clean))
log('duplicates', 'dropped exact duplicate rows', removed)

rows: 8 -> 7
[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [5]:
# Strip formatting characters before converting price text to float.
clean['price'] = (clean['price'].astype(str)
                  .str.replace('$', '', regex=False)
                  .str.replace(',', '', regex=False)
                  .str.strip()
                  .astype(float))

assert clean['price'].dtype == float
log('price', 'stripped dollar signs and whitespace, then converted to float', len(clean))

[price] stripped dollar signs and whitespace, then converted to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [6]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')
missing = int(clean['qty'].isna().sum())
negative = int((clean['qty'] < 0).sum())

# Missing quantities cannot be calculated; negative quantities are retained as refunds.
clean = clean.loc[clean['qty'].notna()].copy()
log('qty', 'dropped rows with missing quantities', missing)
log('qty', 'retained negative quantities as refunds', negative)

[qty] dropped rows with missing quantities (1 row(s))
[qty] retained negative quantities as refunds (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [7]:
print('before:', sorted(clean['category'].unique()))
before_count = clean['category'].nunique()

clean['category'] = (clean['category']
                     .str.replace('-', '', regex=False)
                     .str.lower()
                     .str.strip())

# Apparel and Merch are treated as the same business category.
CATEGORY_MAP = {
    'apparel': 'Merch',
    'merch': 'Merch',
    'food': 'Food',
    'raingear': 'RainGear',
}
clean['category'] = clean['category'].replace(CATEGORY_MAP)
after_count = clean['category'].nunique()

print('after:', sorted(clean['category'].unique()))
log('categories',
    f'normalized labels and mapped Apparel to Merch; {before_count} became {after_count}',
    len(clean))

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after: ['Food', 'Merch', 'RainGear']
[categories] normalized labels and mapped Apparel to Merch; 6 became 3 (6 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [8]:
print('before:', sorted(clean['item'].dropna().unique()))
missing_items = int(clean['item'].isna().sum())

clean['item'] = (clean['item']
                 .str.replace('-', '', regex=False)
                 .str.replace(' ', '', regex=False)
                 .str.lower()
                 .str.strip())

ITEM_MAP = {
    'cheeseburger': 'Cheeseburger',
    'foamfinger': 'Foam Finger',
    'rainponcho': 'Rain Poncho',
    'uvatshirt': 'UVA T-Shirt',
}
clean['item'] = clean['item'].replace(ITEM_MAP).fillna('Unknown')

print('after:', sorted(clean['item'].unique()))
log('items', 'normalized spelling, case, whitespace, and punctuation', len(clean))
log('missing item', "labeled missing item as 'Unknown'", missing_items)

before: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt', 'Unknown']
[items] normalized spelling, case, whitespace, and punctuation (6 row(s))
[missing item] labeled missing item as 'Unknown' (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [9]:
# Coerce unparseable timestamps to NaT, then use the datetime accessor for hour.
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce', format='mixed')
failed = int(clean['ts'].isna().sum())
clean['hour'] = clean['ts'].dt.hour
log('timestamps', 'parsed timestamps and retained failures as NaT', failed)

[timestamps] parsed timestamps and retained failures as NaT (1 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [10]:
# Assertions document the conditions the cleaned table must continue to satisfy.
assert clean.duplicated().sum() == 0
assert clean['price'].dtype == float and clean['price'].notna().all()
assert pd.api.types.is_numeric_dtype(clean['qty']) and clean['qty'].notna().all()
assert set(clean['category'].unique()) <= {'Food', 'Merch', 'RainGear'}
assert clean['item'].notna().all() and 'Unknown' in clean['item'].values
assert pd.api.types.is_datetime64_any_dtype(clean['ts'])
assert clean['ts'].isna().sum() == 1
assert clean['hour'].dropna().between(0, 23).all()

clean['revenue'] = clean['qty'] * clean['price']
print('rows:', len(clean))
print('units:', clean['qty'].sum())
print(f"revenue: ${clean['revenue'].sum():.2f}")
print('distinct categories:', clean['category'].nunique())

rows: 6
units: 7.0
revenue: $88.50
distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [11]:
show_log()

           step                                           decision  rows
0    duplicates                       dropped exact duplicate rows     1
1         price  stripped dollar signs and whitespace, then con...     7
2           qty               dropped rows with missing quantities     1
3           qty            retained negative quantities as refunds     1
4    categories  normalized labels and mapped Apparel to Merch;...     6
5         items  normalized spelling, case, whitespace, and pun...     6
6  missing item                  labeled missing item as 'Unknown'     1
7    timestamps     parsed timestamps and retained failures as NaT     1

**The decision that mattered most:** Step 3: retained the -3 quantity as a refund so revenue reflects returns

**Revenue with it:** $88.5

**Revenue without it:** $106.5

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [12]:
# These values come directly from the completed cleaning pipeline above.
rows_after = len(clean)
revenue_after = clean['revenue'].sum()
biggest_decision = 'Retaining the -3 quantity as a refund reduced revenue by $18.00.'
revenue_other_way = revenue_after + 18.00

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 6
revenue: 88.5
decision that mattered: Retaining the -3 quantity as a refund reduced revenue by $18.00.
revenue the other way: 106.5
